# Asset-level risk assessment

Here goes some text to explain the use case


In [1]:
import warnings
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import shapely
from exactextract import exact_extract
import matplotlib.pyplot as plt
import contextily as cx

from scipy import integrate

from pathlib import Path
import damagescanner.download as download
from damagescanner.core import DamageScanner
from damagescanner.osm import read_osm_data
from damagescanner.config import DICT_CIS_VULNERABILITY_FLOOD

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning) # exactextract gives a warning that is invalid

### Specify the country of interest

Before we continue, we should specify the country for which we want to assess the damage. We use the ISO3 code for the country to download the OpenStreetMap data.

In [56]:
country_full_name = 'Austria'
country_iso3 = 'AUT'

## 2. Loading the Data
In this step, we will prepare and load two key datasets:

1. **Infrastructure data:**  
   This dataset contains information on the location and type of infrastructure (e.g., roads). Each asset may have attributes such as type, length, and replacement cost.

2. **Hazard data:**  
   This dataset includes information on the hazard affecting the infrastructure (e.g., flood depth at various locations).

### Infrastructure Data

We will perform this example analysis for Jamaica. To start the analysis, we first download the OpenStreetMap data from GeoFabrik. 

In [57]:
infrastructure_path = download.get_country_geofabrik(country_iso3)

In [58]:
infrastructure_path

WindowsPath('C:/Users/eks510/osm/osm_pbf/austria-latest.osm.pbf')

In [25]:
features = read_osm_data(infrastructure_path,asset_type='main_roads')

We will not load the data directly, we will let the code itself read the information. It is important, however, to specificy which infrastructure systems you want to include. We do so in the list below:

In [5]:
asset_types = [
#        "roads",
        "rail",
        # "air",
        # "telecom",
        # "education",
        # "healthcare",
        # "power",
    ]

### Hazard Data
For this example, we make use of the flood data provided by [CDRI](https://giri.unepgrid.ch/map?list=explore).

We use a 1/100 flood map to showcase the approach.

In [6]:
hazard_map = xr.open_dataset("https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/CEMS-EFAS/flood_hazard/Europe_RP100_filled_depth.tif", engine="rasterio")

In [7]:
hazard_map

<xarray.Dataset> Size: 23GB
Dimensions:      (band: 1, x: 110162, y: 51992)
Coordinates:
  * band         (band) int32 4B 1
  * x            (x) float64 881kB -24.54 -24.54 -24.54 ... 67.26 67.26 67.26
  * y            (y) float64 416kB 71.13 71.13 71.13 71.13 ... 27.81 27.81 27.81
    spatial_ref  int32 4B ...
Data variables:
    band_data    (band, y, x) float32 23GB ...

### Ancilliary data for processing

In [8]:
NUTS_EU = gpd.read_file("NUTS_RG_20M_2024_3035.geojson").to_crs(4326)

## 3. Preparing the Data

Clip the hazard data to the country of interest.

In [9]:
country_bounds = NUTS_EU.loc[(NUTS_EU.LEVL_CODE == 0) & (NUTS_EU.CNTR_CODE == 'EE')].bounds
country_geom = NUTS_EU.loc[(NUTS_EU.LEVL_CODE == 0) & (NUTS_EU.CNTR_CODE == 'EE')].geometry

In [10]:
country_geom

210    MULTIPOLYGON (((25.72584 59.62819, 25.83016 59...
Name: geometry, dtype: geometry

In [11]:
country_box = shapely.box(country_bounds.minx.values,country_bounds.miny.values,country_bounds.maxx.values,country_bounds.maxy.values)[0]

Set the return periods and download the hazard data

In [23]:
return_periods = [10,20,30,40,50,75,100,200,500]
import rioxarray
data_path = Path(r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\River_floods")
hazard_dict = {}
for return_period in return_periods:
    hazard_map = xr.open_dataset(data_path / f"Europe_RP{return_period}_filled_depth.tif", engine="rasterio")
    #hazard_map = rioxarray.open_rasterio(data_path / f"Europe_RP{return_period}_filled_depth.tif", engine="rasterio")
    hazard_map = hazard_map.rio.clip_box(minx=country_bounds.minx.values[0],
                     miny=country_bounds.miny.values[0],
                     maxx=country_bounds.maxx.values[0],
                     maxy=country_bounds.maxy.values[0]
                    )
    #dim,height, width = hazard_map.shape
    #hazard_map = hazard_map.chunk({'y': 256, 'x': 256})
    hazard_dict[return_period] = hazard_map#.to_dataset(name="band_data")

In [13]:
hazard_map#.to_dataset(name="band_data")

<xarray.Dataset> Size: 78MB
Dimensions:      (band: 1, x: 7660, y: 2533)
Coordinates:
  * band         (band) int32 4B 1
  * x            (x) float64 61kB 21.79 21.79 21.79 21.79 ... 28.17 28.17 28.17
  * y            (y) float64 20kB 59.63 59.63 59.63 59.63 ... 57.52 57.52 57.52
    spatial_ref  int32 4B 0
Data variables:
    band_data    (band, y, x) float32 78MB ...

## 4. Performing the Exposure Assessment
We will use the DamageScanner approach. This is a fully optimised exposure, vulnerability damage and risk calculation method, that can capture a wide range of inputs to perform such an assessment.

In [20]:
asset_type = "roads"
collect_exposure = {}
for return_period in hazard_dict:
    results = DamageScanner(hazard_dict[return_period], features, 
                                    curves=pd.DataFrame(), 
                                    maxdam=pd.DataFrame()
                                    ).exposure(asset_type=asset_type)  
    results[f'coverage_{return_period}'] = results.coverage.apply(lambda x : sum(x))
    results[f'mean_depth_{return_period}'] = results['values'].apply(lambda x : np.mean(x))
    collect_exposure[return_period] = results

convert coverage to meters: 100%|██████████████████████████████████████████████| 13009/13009 [00:10<00:00, 1203.24it/s]


In [26]:
def process_return_period(args):
    """Process a single return period with all calculations."""
    return_period, hazard_data = args
    
    # Calculate damage
    results = DamageScanner(hazard_data, features, 
                                    curves=pd.DataFrame(), 
                                    maxdam=pd.DataFrame()
                                    ).exposure(asset_type=asset_type)  
    
    results[f'coverage_{return_period}'] = results.coverage.apply(lambda x : sum(x))
    results[f'mean_depth_{return_period}'] = results['values'].apply(lambda x : np.mean(x))
    # Return all results for this return period
    
    return return_period, results

In [29]:
import concurrent.futures
from tqdm import tqdm
# Create a list of args to pass to the parallel function
work_items = [(rp, hazard_dict[rp]) for rp in hazard_dict.keys()]

# Use ProcessPoolExecutor for CPU-bound tasks
with concurrent.futures.ProcessPoolExecutor() as executor:
    # Create a tqdm progress bar for the parallel execution
    results = list(tqdm(
        executor.map(process_return_period, work_items),
        total=len(work_items),
        desc="Calculating damages in parallel"
    ))

Calculating damages in parallel:   0%|                                                           | 0/9 [00:00<?, ?it/s]


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
##1:26

## 4. Performing the Damage Assessment
We will use the DamageScanner approach. This is a fully optimised exposure, vulnerability damage and risk calculation method, that can capture a wide range of inputs to perform such an assessment.

In [13]:
maxdam_dict = {key: values[1] for key, values in asset_maxdam_dict.items()}

maxdam = pd.DataFrame.from_dict(maxdam_dict,orient='index').reset_index()
maxdam.columns = ['object_type','damage']

NameError: name 'asset_maxdam_dict' is not defined

In [ ]:
vulnerability_path = "https://zenodo.org/records/10203846/files/Table_D2_Multi-Hazard_Fragility_and_Vulnerability_Curves_V1.0.0.xlsx?download=1"
vul_df = pd.read_excel(vulnerability_path,sheet_name='F_Vuln_Depth').fillna(method='ffill')

In [ ]:
ci_system = DICT_CIS_VULNERABILITY_FLOOD[asset_type]
    
selected_curves = []
for subtype in ci_system:
    selected_curves.append(ci_system[subtype][0])

damage_curves = vul_df[['ID number']+selected_curves]
damage_curves = damage_curves.iloc[4:125,:]
damage_curves.set_index('ID number',inplace=True)
damage_curves.index = damage_curves.index.rename('Depth')  
damage_curves = damage_curves.astype(np.float32)
damage_curves.columns = list(ci_system.keys())
damage_curves = damage_curves.fillna(method='ffill')

In [ ]:
unique_curves = set([x for xs in DICT_CIS_VULNERABILITY_FLOOD[asset_type].values() for x in xs])

In [ ]:
multi_curves = {}
for unique_curve in unique_curves:

    curve_creation = damage_curves.copy()
    
    get_curve_values = vul_df[unique_curve].iloc[4:125].values
    
    for curve in curve_creation:
        curve_creation.loc[:,curve] = get_curve_values

    multi_curves[unique_curve] = curve_creation.astype(np.float32)

In [41]:
protection_standard_map = xr.open_dataset("floodProtection_v2019_paper3.tif", engine="rasterio")

protection_standard_map = protection_standard_map.coarsen(x=10, y=10, boundary="trim").mean()
        
protection_standard_map.rio.write_crs("EPSG:3035", inplace=True)

protection_standard_map = protection_standard_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

In [51]:
NUTS_EU_3035 = NUTS_EU.to_crs(3035)
country_bounds_3035 = NUTS_EU_3035.loc[(NUTS_EU_3035.LEVL_CODE == 0) & (NUTS_EU_3035.CNTR_CODE == 'EE')].bounds

protection_standard_map = protection_standard_map.rio.clip_box(
                    minx=country_bounds_3035.minx.values[0],
                    miny=country_bounds_3035.miny.values[0],
                    maxx=country_bounds_3035.maxx.values[0],
                    maxy=country_bounds_3035.maxy.values[0]
                ).load()

In [53]:
protection_standard_map

<xarray.Dataset> Size: 14kB
Dimensions:      (band: 1, x: 72, y: 47)
Coordinates:
  * band         (band) int32 4B 1
  * x            (x) float64 576B 5.008e+06 5.013e+06 ... 5.358e+06 5.363e+06
  * y            (y) float64 376B 4.172e+06 4.167e+06 ... 3.947e+06 3.942e+06
    spatial_ref  int32 4B 0
Data variables:
    band_data    (band, y, x) float32 14kB 150.0 150.0 150.0 ... 50.0 50.0 48.77

In [54]:
asset_type = 'rail'
protection_standard = DamageScanner(protection_standard_map, features, 
                                    curves=pd.DataFrame(), 
                                    maxdam=pd.DataFrame()
                                    ).exposure(asset_type=asset_type)  

Overlay raster with vector: 100%|██████████████████████████████████████████████████████| 40/40 [01:31<00:00,  2.30s/it]


In [ ]:
protection_standard['design_standard'] = protection_standard['values'].apply(lambda x: np.max(x))

In [ ]:
# Create lookup dictionary for protection standards
asset_region_ds = dict(zip(protection_standard.osm_id.values, protection_standard.design_standard.values))



In [ ]:
# Original damage calculation
asset_type = "rail"
risk = {}
for return_period in hazard_dict:
    results = DamageScanner(hazard_dict[return_period], infrastructure_path, 
                            curves=damage_curves, 
                            maxdam=maxdam
                            ).calculate(asset_type=asset_type, multi_curves=multi_curves)  
    results[return_period] = results[list(multi_curves.keys())].mean(axis=1)
    risk[return_period] = results

In [ ]:
# Simplified risk estimation code accounting for protection standards
RP_list = sorted(list(hazard_dict.keys()))

# Collect the risk for each RP
df_risk = pd.concat(risk, axis=1)

# Get the dataframe of the largest RP
largest_rp = df_risk.loc[:, pd.IndexSlice[RP_list[-1], :]]

# Convert return periods to probabilities
RPS = [1 / x for x in RP_list]

# Utility function for linear interpolation
def interpolate_damage(rp_values, damage_values, target_rp):
    """
    Interpolate damage value for a specific return period.
    
    Args:
        rp_values: List of return periods
        damage_values: List of corresponding damage values
        target_rp: Target return period to interpolate
        
    Returns:
        Interpolated damage value
    """
    # Find indices where target would be inserted to maintain sorted order
    idx = np.searchsorted(rp_values, target_rp)
    
    # If target is exactly at a return period value
    if idx < len(rp_values) and rp_values[idx] == target_rp:
        return damage_values[idx]
    
    # If target is smaller than smallest RP, use the smallest damage
    if idx == 0:
        return damage_values[0]
    
    # If target is larger than largest RP, use the largest damage
    if idx == len(rp_values):
        return damage_values[-1]
    
    # Otherwise, interpolate between the two closest points
    rp_low, rp_high = rp_values[idx-1], rp_values[idx]
    damage_low, damage_high = damage_values[idx-1], damage_values[idx]
    
    # Linear interpolation formula: y = y1 + (x - x1) * (y2 - y1) / (x2 - x1)
    return damage_low + (target_rp - rp_low) * (damage_high - damage_low) / (rp_high - rp_low)

# Estimate risks with protection standards and interpolation
collect_risks = {}
for curve in multi_curves.keys():
    # Get the risk dataframe for this curve
    subrisk = df_risk.loc[:, pd.IndexSlice[:, curve]].fillna(0)
    
    # Apply protection standards during integration
    risk_values = []
    for idx, row in subrisk.iterrows():
        # Get design standard for this asset (default to 0 if not found)
        design_standard = asset_region_ds.get(lookup_osm_id[idx], 0)
        
        # Get all damage values (we'll filter after interpolation)
        rp_damages = {}
        for rp in RP_list:
            damage = row[rp]
            if not pd.isna(damage).values:
                rp_damages[rp] = damage.values[0]
        
        # If no valid damage values, skip to next asset
        if not rp_damages:
            risk_values.append(0)
            continue
        
        # Sort return periods and damage values
        sorted_rps = sorted(rp_damages.keys())
        sorted_damages = [rp_damages[rp] for rp in sorted_rps]
        
        # If design standard is not exactly at one of our return periods,
        # interpolate the damage at the design standard
        if design_standard > 0 and design_standard not in sorted_rps:
            # Add interpolated damage at design standard
            interpolated_damage = interpolate_damage(sorted_rps, sorted_damages, design_standard)
            
            # Insert the interpolated values in the sorted lists
            insert_idx = np.searchsorted(sorted_rps, design_standard)
            sorted_rps.insert(insert_idx, design_standard)
            sorted_damages.insert(insert_idx, interpolated_damage)
        
        # Filter to only include return periods at or above the design standard
        filtered_rps = [rp for rp in sorted_rps if rp >= design_standard]
        filtered_damages = [rp_damages[rp] if rp in rp_damages else 
                           interpolate_damage(sorted_rps, sorted_damages, rp) 
                           for rp in filtered_rps]
        
        # Convert return periods to probabilities for integration
        filtered_probs = [1/rp for rp in filtered_rps]
        
        # Calculate risk using numerical integration
        if len(filtered_damages) >= 2:
            # Reverse for integration (ascending probabilities)
            risk_value = integrate.simpson(
                y=filtered_damages[::-1], 
                x=filtered_probs[::-1]
            )
        elif len(filtered_damages) == 1:
            # Just one point
            risk_value = filtered_probs[0] * filtered_damages[0]
        else:
            # No valid points
            risk_value = 0
            
        risk_values.append(risk_value)
    
    collect_risks[curve] = risk_values

# Create DataFrame with risk values
all_risks = pd.DataFrame.from_dict(collect_risks)

# Format the final results DataFrame
largest_rp.columns = largest_rp.columns.get_level_values(1)
largest_rp = largest_rp.drop(multi_curves.keys(), axis=1)

# Add calculated risk values
largest_rp.loc[:, multi_curves.keys()] = all_risks.values

# Calculate total risk (mean across all curves)
largest_rp['total_risk'] = largest_rp[list(multi_curves.keys())].mean(axis=1)

In [ ]:
lookup_osm_id = dict(zip(largest_rp.index,largest_rp.osm_id))

In [ ]:
subrisk

In [ ]:
asset_region_ds

In [ ]:
# Get the dataframe of the largest RP
largest_rp_2 = df_risk.loc[:, pd.IndexSlice[RP_list[-1], :]]

# only keep the damage values
df_risk = df_risk.loc[
    :, pd.IndexSlice[RP_list, multi_curves.keys()]
].fillna(0)

RPS = [1 / x for x in RP_list]

# estimate risks
collect_risks = {}

for curve in multi_curves.keys():
    subrisk = df_risk.loc[:, pd.IndexSlice[:, curve]]
    collect_risks[curve] = subrisk.apply(
        lambda x: integrate.simpson(y=x[RP_list][::-1], x=RPS[::-1]), axis=1
    ).values

    # save output when tot_risk returns negative values
    if any(subrisk.min() < 0):
        df_risk.to_csv("df_risk.csv")
        subrisk.to_csv("risk.csv")

all_risks = pd.DataFrame.from_dict(collect_risks)

largest_rp_2.columns = largest_rp_2.columns.get_level_values(1)
largest_rp_2 = largest_rp_2.drop(multi_curves.keys(), axis=1)
largest_rp_2.loc[:, multi_curves.keys()] = all_risks.values
largest_rp_2['total_risk'] = largest_rp_2[list(multi_curves.keys())].mean(axis=1)

In [ ]:
largest_rp_2['total_risk'].sum()

In [ ]:
largest_rp['total_risk'].sum()

In [ ]:
risk_dataframe = largest_rp[['osm_id','geometry','object_type','total_risk']]

In [ ]:
z

In [17]:
df_risk = gpd.read_parquet('PRT_rail_river_risk.parquet')

In [18]:
df_risk.head(5)

,osm_id,geometry,object_type,total_risk,total_exposure,hazard_type,protection_standard,length_10,length_20,length_30,...,length_500,mean_damage_10,mean_damage_20,mean_damage_30,mean_damage_40,mean_damage_50,mean_damage_75,mean_damage_100,mean_damage_200,mean_damage_500
4,107613388,"LINESTRING (-8.65864 41.21472, -8.65893 41.214...",rail,5150.629818,2.040188,river,100.000000,70.002279,222.805656,222.805656,...,394.633904,1.459940e+05,6.360736e+05,6.363735e+05,6.365277e+05,6.366342e+05,6.367786e+05,6.367786e+05,6.367786e+05,6.743794e+05
9,1152750407,"LINESTRING (-8.86098 41.81193, -8.86188 41.8126)",rail,2965.481474,1.362462,river,67.000000,24.069667,31.806268,31.806268,...,105.409902,6.879111e+04,7.853948e+04,8.372773e+04,1.282330e+05,1.499922e+05,2.014168e+05,2.226127e+05,2.475439e+05,2.696626e+05
10,1152750408,"LINESTRING (-8.84955 41.86321, -8.84941 41.863...",rail,22790.899714,8.654719,river,68.064514,492.386506,597.069531,597.069531,...,681.906543,1.151411e+06,1.407356e+06,1.543446e+06,1.586767e+06,1.624760e+06,1.717289e+06,1.759984e+06,1.857646e+06,1.899474e+06
11,1152750410,"LINESTRING (-8.7536 41.92635, -8.75329 41.9267...",rail,30377.163402,11.649775,river,68.000000,571.808940,768.798671,861.548456,...,934.276857,1.247033e+06,1.683868e+06,1.948726e+06,2.061115e+06,2.128478e+06,2.257272e+06,2.337442e+06,2.480514e+06,2.586786e+06
12,1152750412,"LINESTRING (-8.67526 41.98843, -8.67481 41.988...",rail,7572.814001,3.423157,river,69.430000,34.669590,202.677103,202.677103,...,309.652508,7.953691e+04,2.335937e+05,3.656999e+05,4.197202e+05,4.600077e+05,5.203701e+05,5.555283e+05,6.662171e+05,7.843557e+05


In [35]:
sorted_damages = df_risk.loc[60,['mean_damage_1', 'mean_damage_100', 'mean_damage_1000']].values
sorted_probs = [1,1/100,1/1000]

In [36]:
sorted_damages

array([22898.40989361907, 49635.428663348655, 67253.43613398977],
      dtype=object)

In [7]:
np.trapz(y=[1.247033e+06,1.683868e+06,1.948726e+06][::-1], 
         x=[1/10, 1/20, 1/30][::-1])

103544.14166666668

In [10]:
np.trapz(y=[45.202635,89.257271,89.257271][::-1], 
         x=[1, 1/100, 1/1000][::-1])

67.360968909